### H1N1 and Seasonal Flu Vaccine Uptake
Developed by Henry El-Jawhari at the Freie Universität (FU) Berlin and Robert Koch Institute (RKI).
#### 2. Modelling

In [ ]:
# Import packages
import pandas as pd, numpy as np, re, pickle, plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")
pio.renderers.default = "notebook"

In [82]:
# Define data types
data_types = {
    "respondent_id": "Int64"
    , "h1n1_concern": "Int64"
    , "h1n1_knowledge": "Int64"
    , "behavioral_antiviral_meds": "Int64"
    , "behavioral_avoidance": "Int64"
    , "behavioral_face_mask": "Int64"
    , "behavioral_wash_hands": "Int64"
    , "behavioral_large_gatherings": "Int64"
    , "behavioral_outside_home": "Int64"
    , "behavioral_touch_face": "Int64"
    , "doctor_recc_h1n1": "Int64"
    , "doctor_recc_seasonal": "Int64"
    , "chronic_med_condition": "Int64"
    , "child_under_6_months": "Int64"
    , "health_worker": "Int64"
    , "health_insurance": "Int64"
    , "opinion_h1n1_vacc_effective": "Int64"
    , "opinion_h1n1_risk": "Int64"
    , "opinion_h1n1_sick_from_vacc": "Int64"
    , "opinion_seas_vacc_effective": "Int64"
    , "opinion_seas_risk": "Int64"
    , "opinion_seas_sick_from_vacc": "Int64"
    , "age_group": "category"
    , "education": "category"
    , "race": "category"
    , "sex": "category"
    , "income_poverty": "category"
    , "marital_status": "category"
    , "rent_or_own": "category"
    , "employment_status": "category"
    , "hhs_geo_region": "category"
    , "census_msa": "category"
    , "household_adults": "Int64"
    , "household_children": "Int64"
    , "employment_industry": "category"
    , "employment_occupation": "category"
    , "h1n1_vaccine": "Int64"
    , "seasonal_vaccine": "Int64"
    }

In [83]:
# Read in data features and labels
features = pd.read_csv("data/training_set_features.csv", dtype = data_types)
labels = pd.read_csv("data/training_set_labels.csv", dtype = data_types)
# Join together features and labels to create a single dataframe
df = features.merge(labels, how = "left", on = "respondent_id")
# For the purpose of modelling we can drop `Response ID`. 
# Also remove any seasonal vaccine features and target, as we are only focusing on H1N1 vaccine uptake (determined in our EDA).
to_exclude = ["respondent_id"] + [x for x in df.columns if re.search("seas", x) is not None]
df.drop(to_exclude, axis = 1, inplace = True)
df.head()

,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,doctor_recc_h1n1,...,marital_status,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine
0,1,0,0,0,0,0,0,1,1,0,...,Not Married,Own,Not in Labor Force,oxchjgsf,Non-MSA,0,0,NaN,NaN,0
1,3,2,0,1,0,1,0,1,1,0,...,Not Married,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0,0,pxcmvdjn,xgwztkwe,0
2,1,1,0,1,0,0,0,0,0,<NA>,...,Not Married,Own,Employed,qufhixun,"MSA, Not Principle City",2,0,rucpziij,xtkaffoo,0
3,1,1,0,1,0,1,1,0,0,0,...,Not Married,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0,0,NaN,NaN,0
4,2,1,0,1,0,1,1,0,1,0,...,Married,Own,Employed,qufhixun,"MSA, Not Principle City",1,0,wxleyezf,emcorrxb,0


In [84]:
# Predictors and response
y = df["h1n1_vaccine"]
X = df.drop("h1n1_vaccine", axis = 1)

In [ ]:
# Pipelines for baseline models, no parameter tuning
# Define numeric and categorical variables
num_cols = list(X.select_dtypes(include = "number").columns)
cat_cols = list(X.select_dtypes(include = "category").columns)

# Convert numeric columns from integer to float
X[num_cols] = X[num_cols].astype(float)

# 80:20 train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Imput missing numerical values with the most frequent (i.e. mode)
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "most_frequent")),
    ]
)

# Imput missing categorical values with the "missing"
# One-hot encode
categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "constant", fill_value = "missing")),
        ("onehot", OneHotEncoder(handle_unknown = "ignore"))
    ]
)

# Transform our columns
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# Run our pipeline for LogisticRegression, RandomForestClassifier, XGBClassifier, and MLPClassifier
# Default parameter values, random state 42 for reproducibility
# For logistic regression and mlp, max_iter was increased to ensure model convergence
logreg_pipe = Pipeline([("preprocess", preprocessor), ("logreg", LogisticRegression(max_iter = 150, random_state = 42))])
rf_pipe     = Pipeline([("preprocess", preprocessor), ("rf", RandomForestClassifier(random_state = 42))])
xgb_pipe    = Pipeline([("preprocess", preprocessor), ("xgb", XGBClassifier(random_state = 42))])
mlp_pipe    = Pipeline([("preprocess", preprocessor), ("mlp", MLPClassifier(max_iter = 500, random_state = 42))])

In [75]:
# Baseline scores for our models
benchmark = pd.DataFrame({
    "LogReg": [logreg_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "Random Forest": [rf_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "XGBoost": [xgb_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "MLP": [mlp_pipe.fit(X_train, y_train).score(X_test, y_test)]
}, index = ["Mean Accuracy Score"]).round(3)

print("Baseline scores ----")
benchmark.T.sort_values("Mean Accuracy Score", ascending = False)

Baseline scores ----


,Mean Accuracy Score
LogReg,0.838
Random Forest,0.834
XGBoost,0.830
MLP,0.782


In [61]:
# Define parameter grid for RandomForest
rf_param_dist = {
    "rf__n_estimators": [100, 300, 500],
    "rf__max_depth": [None, 10, 30],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"],
    "rf__bootstrap": [True, False],
    "rf__criterion": ["gini", "entropy"],
    "rf__class_weight": [None, "balanced"]
}

# Setup RandomizedSearchCV
rf_random_search = RandomizedSearchCV(
    estimator = rf_pipe,
    param_distributions = rf_param_dist,
    n_iter = 50,
    scoring = "roc_auc",
    cv = 5,
    verbose = 1,
    n_jobs = -1,
    random_state = 42
)

# Fit to training data
rf_random_search.fit(X_train, y_train)

# Best score and parameters
print(f"Best ROC AUC score: {rf_random_search.best_score_:.3f}")
print("Best parameters:", rf_random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best ROC AUC score: 0.831
Best parameters: {'rf__n_estimators': 500, 'rf__min_samples_split': 5, 'rf__min_samples_leaf': 2, 'rf__max_features': 'sqrt', 'rf__max_depth': None, 'rf__criterion': 'entropy', 'rf__class_weight': 'balanced', 'rf__bootstrap': True}


In [62]:
# Define the parameter grid for XGBoost
xgb_param_dist = {
    "xgb__n_estimators": [100, 200, 300, 500],
    "xgb__max_depth": [3, 5, 7, 10],
    "xgb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgb__subsample": [0.6, 0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb__gamma": [0, 0.1, 0.3, 0.5],
    "xgb__reg_alpha": [0, 0.1, 1, 10],
    "xgb__reg_lambda": [0.1, 1, 10, 100],
    "xgb__scale_pos_weight": [1, 2, 5, 10],
    "xgb__min_child_weight": [1, 3, 5, 10]
}

# Setup RandomizedSearchCV
xgb_random_search = RandomizedSearchCV(
    estimator = xgb_pipe,
    param_distributions = xgb_param_dist,
    n_iter = 50,
    scoring = "roc_auc",
    cv = 5,
    verbose = 1,
    n_jobs = -1,
    random_state = 42
)

# Fit to training data
xgb_random_search.fit(X_train, y_train)

# Best score and parameters
print(f"Best ROC AUC score: {xgb_random_search.best_score_:.3f}")
print("Best parameters found:", xgb_random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best ROC AUC score: 0.837
Best parameters found: {'xgb__subsample': 0.8, 'xgb__scale_pos_weight': 1, 'xgb__reg_lambda': 100, 'xgb__reg_alpha': 0.1, 'xgb__n_estimators': 200, 'xgb__min_child_weight': 3, 'xgb__max_depth': 5, 'xgb__learning_rate': 0.1, 'xgb__gamma': 0, 'xgb__colsample_bytree': 1.0}


In [63]:
# Define the parameter grid for MLP
mlp_param_dist = {
    "mlp__hidden_layer_sizes": [(50,), (100,), (50, 50), (100, 100), (50, 100, 50)],
    "mlp__activation": ["relu", "tanh", "logistic"],
    "mlp__solver": ["adam", "sgd"],
    "mlp__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
    "mlp__learning_rate": ["constant", "invscaling", "adaptive"],
    "mlp__learning_rate_init": [0.001, 0.01, 0.05, 0.1],
    "mlp__batch_size": ["auto", 32, 64, 128],
    "mlp__max_iter": [250, 500, 1000],
    "mlp__momentum": [0.8, 0.9, 0.95],
    "mlp__beta_1": [0.8, 0.9, 0.99],
    "mlp__beta_2": [0.9, 0.99, 0.999]
}

# Setup RandomizedSearchCV
mlp_random_search = RandomizedSearchCV(
    estimator = mlp_pipe,
    param_distributions = mlp_param_dist,
    n_iter = 50,
    scoring = "roc_auc",
    cv = 5,
    verbose = 1,
    n_jobs = -1,
    random_state = 42
)

# Fit to training data
mlp_random_search.fit(X_train, y_train)

# Best score and parameters
print(f"Best ROC AUC score: {mlp_random_search.best_score_:.3f}")
print("Best parameters found:", mlp_random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best ROC AUC score: 0.831
Best parameters found: {'mlp__solver': 'adam', 'mlp__momentum': 0.95, 'mlp__max_iter': 500, 'mlp__learning_rate_init': 0.01, 'mlp__learning_rate': 'adaptive', 'mlp__hidden_layer_sizes': (50,), 'mlp__beta_2': 0.999, 'mlp__beta_1': 0.9, 'mlp__batch_size': 32, 'mlp__alpha': 0.01, 'mlp__activation': 'relu'}


In [ ]:
# Pipelines for tuned RandomForest, XGB, and MLP models
rf_tuned_pipe   = Pipeline([("preprocess", preprocessor), ("rf_tuned", RandomForestClassifier(**{x.replace("rf__", ""): v for x, v in rf_random_search.best_params_.items()}))])
xgb_tuned_pipe  = Pipeline([("preprocess", preprocessor), ("xgb_tuned", XGBClassifier(**{x.replace("xgb__", ""): v for x, v in xgb_random_search.best_params_.items()}))])
mlp_tuned_pipe  = Pipeline([("preprocess", preprocessor), ("mlp_tuned", MLPClassifier(**{x.replace("mlp__", ""): v for x, v in mlp_random_search.best_params_.items()}))])

In [65]:
# Updated scores for our hyperparameter tuned models
tuned = pd.DataFrame({
    "LogReg": [logreg_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "Random Forest": [rf_tuned_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "XGBoost": [xgb_tuned_pipe.fit(X_train, y_train).score(X_test, y_test)],
    "MLP": [mlp_tuned_pipe.fit(X_train, y_train).score(X_test, y_test)]
}, index = ["Mean Accuracy Score"]).round(3)

print("Hyperparameter tuned scores ----")
tuned.T.sort_values("Mean Accuracy Score", ascending = False)

Hyperparameter tuned scores ----


,Mean Accuracy Score
XGBoost,0.840
LogReg,0.838
MLP,0.834
Random Forest,0.832


After hyperparameter-tuning, the XGBoost model performs the best with the highest mean accuracy score. This is the model we select for deployment.

In [76]:
# Hyperparameter XGBoost has the best score. Select this model for deployment
# Generate predictions
predictions = pd.DataFrame({
        "xgb_tuned": xgb_tuned_pipe.fit(X_train, y_train).predict_proba(X_test)[:,1],
        "actual": list(y_test)
        }).melt(id_vars = "actual", var_name = "model", value_name = "predicted_proba")

# Binarise our predicted probabilities with a threshold of 0.5
predictions["predicted_binary"] = np.where(predictions["predicted_proba"] >= 0.5, 1, 0)
predictions.head()

,actual,model,predicted_proba,predicted_binary
0,0,xgb_tuned,0.143417,0
1,0,xgb_tuned,0.106026,0
2,0,xgb_tuned,0.042697,0
3,0,xgb_tuned,0.158882,0
4,0,xgb_tuned,0.158459,0


In [86]:
# Save data and the hyper-parameter tuned XGB model into a dictionary
modelling_output = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
    "y_test": y_test,
    "xgb_tuned_model": xgb_tuned_pipe,
    "predictions": predictions
}

In [87]:
#  Export as a pickle file
with open("modelling_output.pickle", "wb") as file:
    pickle.dump(modelling_output, file, protocol = pickle.HIGHEST_PROTOCOL)